<a href="https://colab.research.google.com/github/Kamsi08/Academic-Advisor-System/blob/main/academic_advisor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle, os

In [3]:
np.random.seed(42)
n = 1000

data = {
    # Academic profile
    'gpa': np.round(np.random.uniform(1.5, 4.0, n), 2),
    'year': np.random.randint(1, 5, n),                 # 1=Freshman … 4=Senior
    'credits_completed': np.random.randint(0, 130, n),

    # Skill / performance
    'math_score': np.random.randint(40, 100, n),
    'programming_score': np.random.randint(30, 100, n),
    'communication_score': np.random.randint(40, 100, n),

    # Career / academic interests (multi-select)
    'interest_ai': np.random.randint(0, 2, n),
    'interest_web': np.random.randint(0, 2, n),
    'interest_data': np.random.randint(0, 2, n),
    'interest_research': np.random.randint(0, 2, n),

    # Risk / wellness flags
    'on_probation': np.random.randint(0, 2, n),
    'financial_need': np.random.randint(0, 2, n),
    'mental_health_flag': np.random.randint(0, 2, n),
    'at_risk': np.random.randint(0, 2, n),
}

df = pd.DataFrame(data)

In [4]:
def advise(row):
    # 1. Immediate intervention — monitoring + referral
    if row['on_probation'] == 1 or row['at_risk'] == 1 or row['gpa'] < 2.0:
        return 'Intervention & Support'
    # 2. Wellness / resource referral
    if row['mental_health_flag'] == 1 or row['financial_need'] == 1:
        return 'Resource Referral'
    # 3. Career & goal guidance
    if row['year'] >= 3 and row['gpa'] >= 3.0:
        if row['interest_research'] == 1:
            return 'Research & Grad School Track'
        if row['interest_ai'] == 1 and row['math_score'] > 75:
            return 'AI/ML Career Track'
        if row['interest_data'] == 1:
            return 'Data Science Career Track'
        if row['interest_web'] == 1 and row['programming_score'] > 70:
            return 'Web Development Career Track'
    # 4. Course selection & registration
    if row['credits_completed'] < 30:
        return 'Foundational Course Planning'
    if row['year'] == 2:
        return 'Major Declaration Guidance'
    # 5. Default — academic planning
    return 'Academic Planning'

df['advising_action'] = df.apply(advise, axis=1)
df['advising_action'].value_counts()

,count
advising_action,
Intervention & Support,811
Resource Referral,143
Academic Planning,20
Major Declaration Guidance,10
Foundational Course Planning,9
Data Science Career Track,3
Research & Grad School Track,3
Web Development Career Track,1


In [6]:
X = df.drop('advising_action', axis=1)
y = df['advising_action']

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.955
              precision    recall  f1-score   support

           0       0.40      0.40      0.40         5
           2       0.00      0.00      0.00         4
           3       0.99      1.00      1.00       162
           4       0.33      1.00      0.50         1
           5       0.00      0.00      0.00         2
           6       0.93      1.00      0.96        26

    accuracy                           0.95       200
   macro avg       0.44      0.57      0.48       200
weighted avg       0.94      0.95      0.95       200



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [7]:
with open('academic_advisor_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

for name in ['academic_advisor_model.pkl', 'label_encoder.pkl']:
    print(name, os.path.getsize(name), "bytes")

academic_advisor_model.pkl 2258169 bytes
label_encoder.pkl 458 bytes


In [11]:
from google.colab import files
files.download('academic_advisor_model.pkl')
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>